# nb_11_gold_customer_360

**Layer**: Gold | **Source**: `lh_silver_insurance` | **Target**: `lh_gold_insurance.gold_customer_360`

**Purpose**: Build a 360-degree customer view combining policies, premiums, and claims data.

**Output Columns**:
| Column | Description |
|---|---|
| customer_id | Unique customer identifier |
| customer_name | Full name (first + last) |
| city, state | Customer location |
| customer_since | Date customer was onboarded |
| total_policies | Count of policies held |
| active_policies | Count of active policies |
| total_annual_premium | Sum of annual premium amounts |
| total_claims | Count of claims filed |
| open_claims | Count of open/under_review claims |
| total_claim_amount | Sum of estimated claim amounts |
| total_paid_premiums | Sum of premium payments made |
| customer_risk_score | Simple risk indicator (claims/policies ratio) |

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, concat_ws, count, sum as _sum, round as _round,
    current_timestamp, when, coalesce, lit
)

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_11_gold_customer_360").getOrCreate()

print("nb_11_gold_customer_360 - Gold Layer Aggregation")

In [ ]:
# ============================================================
# Step 1: Read Silver tables
# ============================================================
df_customers = spark.table("customers")
df_policies = spark.table("policies")
df_premiums = spark.table("premiums")
df_claims = spark.table("claims")

print(f"Customers: {df_customers.count():,}")
print(f"Policies:  {df_policies.count():,}")
print(f"Premiums:  {df_premiums.count():,}")
print(f"Claims:    {df_claims.count():,}")

In [ ]:
# ============================================================
# Step 2: Aggregate policies per customer
# ============================================================
df_policy_agg = df_policies.groupBy("customer_id").agg(
    count("policy_id").alias("total_policies"),
    count(when(col("status") == "active", True)).alias("active_policies"),
    _round(_sum("annual_premium"), 2).alias("total_annual_premium")
)

print(f"Policy aggregation: {df_policy_agg.count():,} customers with policies")

In [ ]:
# ============================================================
# Step 3: Aggregate premiums per customer (via policy)
# ============================================================
df_premium_agg = df_premiums.join(
    df_policies.select("policy_id", "customer_id"),
    "policy_id",
    "inner"
).groupBy("customer_id").agg(
    _round(_sum("amount_paid"), 2).alias("total_paid_premiums")
)

print(f"Premium aggregation: {df_premium_agg.count():,} customers with premiums")

In [ ]:
# ============================================================
# Step 4: Aggregate claims per customer (via policy)
# ============================================================
df_claim_agg = df_claims.join(
    df_policies.select("policy_id", "customer_id"),
    "policy_id",
    "inner"
).groupBy("customer_id").agg(
    count("claim_id").alias("total_claims"),
    count(when(col("claim_status").isin("open", "under_review"), True)).alias("open_claims"),
    _round(_sum("estimated_amount"), 2).alias("total_claim_amount")
)

print(f"Claim aggregation: {df_claim_agg.count():,} customers with claims")

In [ ]:
# ============================================================
# Step 5: Build Customer 360 view
# ============================================================
df_gold = df_customers \
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name"))) \
    .select("customer_id", "customer_name", "city", "state", "customer_since") \
    .join(df_policy_agg, "customer_id", "left") \
    .join(df_premium_agg, "customer_id", "left") \
    .join(df_claim_agg, "customer_id", "left") \
    .fillna(0, ["total_policies", "active_policies", "total_annual_premium",
                "total_paid_premiums", "total_claims", "open_claims", "total_claim_amount"]) \
    .withColumn(
        "customer_risk_score",
        _round(
            when(col("total_policies") > 0,
                 col("total_claims") / col("total_policies")
            ).otherwise(0.0), 2
        )
    ).orderBy("customer_id")

gold_count = df_gold.count()
print(f"Gold customer 360: {gold_count:,} rows")
df_gold.show(5, truncate=20)

In [ ]:
# ============================================================
# Step 6: Write to Gold
# ============================================================
df_gold.withColumn("_processed_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_360")

print(f"✓ {gold_count:,} rows written to gold_customer_360")

In [ ]:
# ============================================================
# Validation
# ============================================================
readback = spark.table("gold_customer_360").count()
assert readback == gold_count
print(f"✓ Verification passed: {readback:,} rows")

# Top 10 highest-risk customers
print("\n--- Top 10 Highest Risk Customers ---")
spark.sql("""
    SELECT customer_id, customer_name, total_policies, total_claims,
           total_claim_amount, customer_risk_score
    FROM gold_customer_360
    WHERE total_policies > 0
    ORDER BY customer_risk_score DESC
    LIMIT 10
""").show(truncate=False)